# TC3 — Respostas às perguntas norteadoras

**Síntese executiva pronta para avaliação e apresentação.** As conclusões abaixo usam os resultados já executados no notebook completo. Os gráficos são sínteses dos resultados existentes; não refazem o pipeline.

## 1. É possível estimar a alfabetização com os dados disponíveis?

**Resposta: sim, com capacidade preditiva moderada — não suficiente para tratar a previsão como diagnóstico definitivo.** O Random Forest final obteve ROC AUC **0,6705**, Average Precision **0,6841**, acurácia balanceada **0,6196** e F1 **0,6392** no teste independente. Há sinal preditivo, mas também erro relevante; o modelo deve apoiar priorização e investigação, não substituir indicadores oficiais.

*Referência no notebook completo: avaliação final independente, métricas, calibração e curvas ROC/Precisão-Recall.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metricas = pd.DataFrame({
    'Métrica': ['ROC AUC', 'Average Precision', 'Acurácia balanceada', 'F1'],
    'Valor': [0.670483, 0.684097, 0.619615, 0.639195]
})
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(metricas['Métrica'], metricas['Valor'])
ax.set_xlim(0, 1); ax.set_xlabel('Valor (0 a 1)')
ax.set_title('Desempenho do modelo no teste independente')
for i, v in enumerate(metricas['Valor']): ax.text(v + .01, i, f'{v:.3f}', va='center')
plt.tight_layout(); plt.show()
display(metricas)

## 2. Quais fatores aparecem mais associados às estimativas?

**Resposta: a UF é, de longe, a variável mais importante no modelo; depois aparecem Bolsa Família, INSE, rede de ensino e infraestrutura.** A importância de `sigla_uf` é muito superior às demais. Isso pode refletir diferenças territoriais e também atuar como proxy de fatores não observados; portanto, não significa que a UF cause o resultado. A comparação sem UF indica que a performance muda pouco e não melhora de forma uniforme quando a UF é incluída.

*Referência: importância por permutação, interpretação responsável de UF e análise de sensibilidade com/sem UF.*

In [ ]:
fatores = pd.DataFrame({
    'Variável': ['UF', 'Bolsa Família', 'INSE médio', 'Rede', 'Laboratório de informática'],
    'Importância por permutação': [0.081454, 0.013114, 0.008716, 0.007759, 0.005250]
})
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fatores['Variável'][::-1], fatores['Importância por permutação'][::-1])
ax.set_xlabel('Queda média na métrica ao permutar a variável')
ax.set_title('Cinco variáveis com maior importância por permutação')
plt.tight_layout(); plt.show()
display(fatores)

## 3. A infraestrutura escolar está associada aos resultados?

**Resposta: há associação descritiva, mas ela não é uniforme e não demonstra causalidade.** Para a faixa de bibliotecas/salas de leitura, a taxa observada aumenta entre as faixas mais baixas e mais altas em 2023 (de **45,0%** para **53,4%**) e em 2024 (de **48,2%** para **57,0%**). A análise completa também considera laboratórios, quadras, água, energia, esgoto, internet e banda larga. Como os dados são agregados, não é possível atribuir o resultado de um aluno à infraestrutura de sua escola específica.

*Referência: análise contextual de infraestrutura, por faixas e ano.*

In [ ]:
infra = pd.DataFrame({
    'Faixa de infraestrutura': ['Mais baixa', 'Faixa 2', 'Faixa 3', 'Faixa 4', 'Mais alta'],
    '2023': [0.449656, 0.494824, 0.509408, 0.510740, 0.534488],
    '2024': [0.481904, 0.513073, 0.533124, 0.518753, 0.569920]
})
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(infra['Faixa de infraestrutura'], infra['2023']*100, marker='o', label='2023')
ax.plot(infra['Faixa de infraestrutura'], infra['2024']*100, marker='o', label='2024')
ax.set_ylabel('Taxa observada de alfabetização (%)'); ax.set_title('Biblioteca/sala de leitura: taxa observada por faixa')
ax.legend(); ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()
display(infra)

## 4. O INSE municipal está associado à alfabetização?

**Resposta: a associação com a taxa observada é fraca; a relação com a probabilidade estimada pelo modelo é mais forte, sobretudo em 2023.** A correlação de Spearman entre INSE e taxa observada foi **0,059 em 2023** e **0,024 em 2024**. Com a probabilidade prevista, foi **0,281 em 2023** e **0,174 em 2024**. O INSE usado é uma medida contextual municipal de 2023, reaproveitada na comparação de 2024; não representa o INSE individual de cada estudante nem uma mudança temporal do indicador.

*Referência: faixas de INSE e tabela de correlações.*

In [ ]:
inse = pd.DataFrame({
    'Ano': [2023, 2024],
    'Spearman: taxa observada': [0.058576, 0.024030],
    'Spearman: probabilidade prevista': [0.280520, 0.173926]
})
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(inse)); w = 0.34
ax.bar([i-w/2 for i in x], inse['Spearman: taxa observada'], width=w, label='Taxa observada')
ax.bar([i+w/2 for i in x], inse['Spearman: probabilidade prevista'], width=w, label='Probabilidade prevista')
ax.set_xticks(list(x), inse['Ano']); ax.set_ylabel('Correlação de Spearman')
ax.set_title('INSE municipal e alfabetização'); ax.legend()
plt.tight_layout(); plt.show()
display(inse)

## 5. O desempenho municipal se mantém de 2023 para 2024?

**Resposta:** há persistência moderada na ordenação dos resultados observados e persistência alta na ordenação das probabilidades previstas. Entre **5.653 pares município/rede comparáveis**, a correlação de Spearman dos resultados observados foi **0,447**; para as probabilidades previstas, **0,894**. A diferença indica que as previsões são mais estáveis entre os anos do que os resultados reais observados. Isso não prova que o modelo represente adequadamente as mudanças ocorridas: parte da estabilidade pode decorrer das características territoriais persistentes e a cobertura entre anos não é idêntica. A análise cobre apenas dois anos e não inclui São Paulo no conjunto comparável de 2023.

*Referência: painel municipal 2023–2024 e transição entre faixas; notebook monstro, seção 15.

In [ ]:
persist = pd.DataFrame({
    'Comparação': ['Resultados observados', 'Probabilidades previstas'],
    'Spearman': [0.447454, 0.893588]
})
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(persist['Comparação'][::-1], persist['Spearman'][::-1])
ax.set_xlim(0, 1); ax.set_xlabel('Correlação de Spearman')
ax.set_title('Persistência entre 2023 e 2024 (5.653 pares município/rede)')
for i, v in enumerate(persist['Spearman'][::-1]): ax.text(v+.01, i, f'{v:.3f}', va='center')
plt.tight_layout(); plt.show()
display(persist)

## 6. Finanças públicas e condições socioeconômicas ajudam a explicar diferenças?

**Resposta:** as variáveis financeiras ajudam a contextualizar as diferenças entre municípios, mas esta análise não identifica efeitos causais nem permite concluir que aumentar isoladamente uma rubrica elevará a alfabetização. Bolsa Família aparece entre as variáveis mais importantes do modelo; Fundeb, receitas municipais e PIB per capita são comparados por faixas e ano no gráfico abaixo. Valores totais podem refletir o porte populacional, e não equivalem a gasto por aluno ou qualidade de execução.

O gráfico usa a taxa observada exportada no CSV financeiro. As diferenças entre faixas são descritivas e não devem ser interpretadas como efeito financeiro isolado.

*Referência da análise completa: notebook monstro, seção 16; arquivo `07_financas_por_faixa_ano.csv`.

In [ ]:
from pathlib import Path

# Localiza o CSV já exportado pelo pipeline, sem refazer a análise.
candidatos = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in candidatos if (p / 'resultados' / 'tabelas' / '07_financas_por_faixa_ano.csv').is_file()), None)
if raiz is None:
    raise FileNotFoundError("Não encontrei resultados/tabelas/07_financas_por_faixa_ano.csv. Abra o notebook a partir da raiz do repositório ou ajuste o caminho abaixo.")

financas = pd.read_csv(raiz / 'resultados' / 'tabelas' / '07_financas_por_faixa_ano.csv')
necessarias = {'feature', 'ano_referencia', 'faixa', 'taxa_observada'}
faltantes = necessarias - set(financas.columns)
if faltantes:
    raise ValueError(f"Colunas esperadas ausentes no CSV financeiro: {sorted(faltantes)}. Colunas disponíveis: {financas.columns.tolist()}")

# Mantém apenas as quatro dimensões financeiras/contextuais citadas na resposta.
mapa_features = {
    'valor_bolsa_familia__signed_log1p': 'Bolsa Família',
    'valor_fundeb__signed_log1p': 'Fundeb',
    'valor_receitas__signed_log1p': 'Receitas municipais',
    'valor_pib_per_capita__signed_log1p': 'PIB per capita',
}
financas['variavel'] = financas['feature'].map(mapa_features)
fin_plot = financas[financas['variavel'].notna()].copy()
if fin_plot.empty:
    print('Os nomes das variáveis no CSV diferem dos nomes previstos. Variáveis disponíveis:')
    display(financas[['feature']].drop_duplicates().sort_values('feature'))
    raise ValueError('Ajuste mapa_features aos nomes reais listados acima; nenhum valor foi inventado.')

# Ordena faixas de acordo com a ordem existente no CSV.
anos = sorted(fin_plot['ano_referencia'].dropna().unique())
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, (feature, nome) in zip(axes.flat, mapa_features.items()):
    d = fin_plot[fin_plot['feature'] == feature]
    if d.empty:
        ax.set_visible(False)
        continue
    for ano in anos:
        da = d[d['ano_referencia'] == ano]
        if not da.empty:
            ax.plot(da['faixa'].astype(str), da['taxa_observada'] * 100, marker='o', label=str(int(ano)))
    ax.set_title(nome)
    ax.set_ylabel('Taxa observada (%)')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', alpha=.25)
    ax.legend(title='Ano')
fig.suptitle('Taxa observada por faixa de variável financeira/contextual', y=1.02)
plt.tight_layout()
plt.show()
display(fin_plot[['variavel', 'ano_referencia', 'faixa', 'registros', 'feature_media', 'taxa_observada', 'probabilidade_media']].sort_values(['variavel', 'ano_referencia', 'faixa']))

## 7. É possível agrupar municípios em perfis distintos?

**Resposta: sim, exploratoriamente, mas os grupos não devem ser tratados como categorias definitivas.** A solução com **3 clusters** apresentou o maior silhouette entre as alternativas testadas (**0,241**), resultado baixo e indicativo de separação limitada entre os grupos. Um dos clusters tem apenas **25 registros município-ano**, muito menor que os outros; esse grupo pequeno precisa ser interpretado com cautela. Os perfis servem como apoio à exploração e à priorização, não como classificação oficial.

*Referência: seleção de clusters e perfil dos grupos.*

In [ ]:
sil = pd.DataFrame({'Número de clusters (k)': [2,3,4,5,6], 'Silhouette': [0.236435,0.240664,0.239402,0.200605,0.196756]})
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(sil['Número de clusters (k)'], sil['Silhouette'], marker='o')
ax.set_xticks(sil['Número de clusters (k)']); ax.set_ylabel('Silhouette')
ax.set_title('Seleção do número de clusters'); ax.annotate('k=3 escolhido', (3,0.240664), xytext=(3.5,0.23), arrowprops={'arrowstyle':'->'})
plt.tight_layout(); plt.show()
display(sil)
display(pd.DataFrame({'Cluster': [0,1,2], 'Registros município-ano': [5058,7255,25], 'Interpretação': ['Perfil amplo', 'Perfil amplo', 'Grupo muito pequeno; cautela']}))

## 8. Quais municípios podem não alcançar as metas futuras?

**Resposta: o pipeline produziu uma classificação de risco em relação às metas, mas ela deve ser apresentada como sinalização de planejamento, não como previsão garantida.** A cobertura da meta municipal foi de **4.664 municípios em 2023** e **5.327 em 2024**; ficaram sem meta correspondente **145 em 2023** e **99 em 2024**. Os riscos foram calculados comparando o indicador estimado com metas futuras de 2024–2030. A ausência de meta correspondente limita a cobertura, e a previsão depende das hipóteses e da qualidade dos dados disponíveis.

*Referência: risco municipal por metas e auditoria de cobertura.*

In [ ]:
cobertura = pd.DataFrame({
    'Ano de referência': [2023, 2023, 2024, 2024],
    'Cobertura': ['Com meta', 'Sem meta', 'Com meta', 'Sem meta'],
    'Municípios': [4664, 145, 5327, 99]
})
fig, ax = plt.subplots(figsize=(7.5, 4))
categorias = ['Com meta', 'Sem meta']
x = range(len(categorias)); largura = 0.34
for deslocamento, ano in [(-largura/2, 2023), (largura/2, 2024)]:
    d = cobertura[cobertura['Ano de referência'] == ano].set_index('Cobertura').reindex(categorias)
    barras = ax.bar([i + deslocamento for i in x], d['Municípios'], largura, label=str(ano))
    ax.bar_label(barras, padding=3, fmt='%.0f')
ax.set_xticks(list(x), categorias)
ax.set_ylabel('Quantidade de municípios')
ax.set_title('Cobertura das metas municipais por ano')
ax.legend(title='Ano')
ax.set_ylim(0, 6000)
plt.tight_layout(); plt.show()
display(cobertura)

## Conclusão executiva

1. **Predição:** possível, com desempenho moderado (ROC AUC 0,670).
2. **Fatores:** UF domina a importância; Bolsa Família, INSE, rede e laboratório aparecem em seguida. Importância não é causalidade.
3. **Infraestrutura:** as taxas observadas tendem a ser maiores nas faixas mais altas de biblioteca/sala de leitura, com variação entre anos e faixas.
4. **INSE:** correlação fraca com a taxa observada; moderada com a probabilidade estimada em 2023 e menor em 2024.
5. **Histórico municipal:** resultados observados têm persistência moderada (rho 0,447); previsões, persistência alta (rho 0,894).
6. **Finanças:** oferecem contexto, mas os dados não isolam efeito causal nem gasto por aluno.
7. **Clusters:** três grupos são uma solução exploratória, com separação fraca e um grupo muito pequeno.
8. **Metas futuras:** há classificação de risco e cobertura documentada, mas não é garantia de resultado futuro.

**Limite transversal:** os dados são agregados/contextuais em várias análises; não sustentam conclusões individuais sobre alunos nem afirmações causais.